# X-OPT - STRUCTURAL INTERPRETABILITY BENCHMARK

This notebook benchmarks the post-hoc structural interpretability pipeline over the OR-Library p-median instances. For each instance, the notebook performs the following steps:

1. run the metaheuristic;
2. build the long-term memory (LTM);
3. keep the top 5% best LTM solutions;
4. build the weighted co-occurrence graph;
5. extract communities, Max-p-Cut, k-core, and densest subgraph;
6. compute the requested interpretability metrics;
7. display and save the final CSV table.

The notebook also stores:

- the metaheuristic runtime from step 1; and
- the structure-extraction runtime from step 5,

so the runtime cost of interpretability can be evaluated directly.

### SETUP

Adding a new path to the Python interpreter:

In [1]:
import sys

from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "instances").exists() and \
           (candidate / "pymedian" ).exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing 'instances' and 'pymedian'."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

PROJECT_ROOT = /home/rei-luisinho/xopt


Importing the libraries:

In [2]:
from __future__ import annotations

import re
import math
import pymedian

import numpy    as np
import pandas   as pd
import networkx as nx

from tqdm.auto           import tqdm
from networkx.algorithms import community
from itertools           import combinations
from time                import perf_counter

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

Defining helper functions:

In [3]:
def instance_sort_key(pathlike: str | Path) -> tuple[int, str]:
    stem  = Path     (pathlike ).stem
    match = re.search(r"(\d+)$", stem)

    if match is None:
        return (10**9, stem)

    return (int(match.group(1)), stem)


def canonical_instance_name(instance_name: str) -> str:
    instance_name = instance_name.strip()

    if instance_name.endswith(".txt"):
        return instance_name

    return f"{instance_name}.txt"


def list_orlibrary_instances(instances_dir: Path) -> list[str]:
    return sorted(
        [
            path.name
            for path in instances_dir.glob("pmed*.txt")
            if  path.name != "pmedopt.txt"
        ],
        key=instance_sort_key,
    )


def read_instance_metadata(instance_path: Path) -> dict[str, int]:
    header = instance_path.read_text().splitlines()[0].split()

    if len(header) < 3:
        raise ValueError(
            f"Could not parse instance header: {instance_path}"
        )

    return {
        "n": int(header[0]),
        "p": int(header[2]),
    }


def load_best_known_costs(pmedopt_path: Path) -> pd.DataFrame:
    rows = []

    for raw_line in pmedopt_path.read_text().splitlines()[1:]:
        line = raw_line.strip()
        if not line:
            continue

        parts = line.split()

        rows.append(
            {
                "instance_id"     : parts[0].strip(),
                "best_known_cost" : float(parts[1] ),
            }
        )

    df = pd.DataFrame(rows)

    df["instance_order"] = df["instance_id"].map(
        lambda value: instance_sort_key(value)[0]
    )

    return (
        df.sort_values(["instance_order", "instance_id"])
          .drop       (columns="instance_order"         )
          .reset_index(drop   =True)
    )


def compute_gap_percent(
    cost            : float | None,
    best_known_cost : float | None,
) -> float:
    if cost is None or pd.isna(cost):
        return np.nan

    if best_known_cost  is None or \
       pd.isna(best_known_cost) or \
       float  (best_known_cost) == 0.0:
        return np.nan

    return 100.0 * (float(cost) - float(best_known_cost)) / float(best_known_cost)

### EXPERIMENT CONFIGURATION

In [4]:
INSTANCES_DIR = PROJECT_ROOT  / "instances"
PMEDOPT_PATH  = INSTANCES_DIR / "pmedopt.txt"

OUTPUT_DIR   = PROJECT_ROOT / "notebooks" / "experiments_sbpo" / "artifacts"
RESULTS_CSV  = OUTPUT_DIR   / "structural_interpretability_benchmark.csv"
FAILURES_CSV = OUTPUT_DIR   / "structural_interpretability_benchmark_failures.csv"

DEFAULT_INSTANCE_NAMES = list_orlibrary_instances(INSTANCES_DIR)
INSTANCE_NAMES         = DEFAULT_INSTANCE_NAMES


RESTARTS = 8
MAX_ITER = 25
FACTOR   = 1
DETAILS_FORMAT = "binary"

TOP_FRACTION = 0.05

GLOBAL_SEED        = 42
MAX_P_CUT_RESTARTS = 30
MAX_P_CUT_MAX_ITER = 2000


SAVE_RESULTS_CSV  = True
SAVE_FAILURES_CSV = True

BEST_KNOWN_COSTS_DF = load_best_known_costs(PMEDOPT_PATH)
BEST_KNOWN_COSTS    = BEST_KNOWN_COSTS_DF.set_index("instance_id")["best_known_cost"].to_dict()

Verifying the hyperparameters:

In [5]:
print(f"Project root        : {PROJECT_ROOT }")
print(f"Instances folder    : {INSTANCES_DIR}")
print(f"Results CSV         : {RESULTS_CSV  }")

Project root        : /home/rei-luisinho/xopt
Instances folder    : /home/rei-luisinho/xopt/instances
Results CSV         : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/structural_interpretability_benchmark.csv


In [8]:
print(f"Number of instances            : {len(INSTANCE_NAMES)}")
print(f"Top fraction kept from the LTM : {TOP_FRACTION   :.0%}")
print(f"Metaheuristic parameters       : restarts={RESTARTS       :2d}, max_iter={MAX_ITER       :4d}, factor={FACTOR     }")
print(f"Max-p-Cut parameters           : restarts={MAX_P_CUT_RESTARTS}, max_iter={MAX_P_CUT_MAX_ITER}, seed  ={GLOBAL_SEED}")

Number of instances            : 40
Top fraction kept from the LTM : 5%
Metaheuristic parameters       : restarts= 8, max_iter=  25, factor=1
Max-p-Cut parameters           : restarts=30, max_iter=2000, seed  =42


In [9]:
selected_instances_df = pd.DataFrame(
    [
        {
            "instance"        :      canonical_instance_name(instance_name)      ,
            "instance_id"     : Path(canonical_instance_name(instance_name)).stem,
            "best_known_cost" : BEST_KNOWN_COSTS.get(
                Path(canonical_instance_name(instance_name)).stem, np.nan,
            ),
            **read_instance_metadata(INSTANCES_DIR / canonical_instance_name(instance_name)),
        }
        for instance_name in INSTANCE_NAMES
    ]
)

selected_instances_df["instance_order"] = selected_instances_df["instance"].map(
    lambda value: instance_sort_key(value)[0]
)

selected_instances_df = (
    selected_instances_df.sort_values(["instance_order", "instance"])
                         .drop       (columns="instance_order"      )
                         .reset_index(drop   =True)
)

display(selected_instances_df.head(10))

,instance,instance_id,best_known_cost,n,p
0,pmed1.txt,pmed1,5819.0,100,5
1,pmed2.txt,pmed2,4093.0,100,10
2,pmed3.txt,pmed3,4250.0,100,10
3,pmed4.txt,pmed4,3034.0,100,20
4,pmed5.txt,pmed5,1355.0,100,33
5,pmed6.txt,pmed6,7824.0,200,5
6,pmed7.txt,pmed7,5631.0,200,10
7,pmed8.txt,pmed8,4445.0,200,20
8,pmed9.txt,pmed9,2734.0,200,40
9,pmed10.txt,pmed10,1255.0,200,67


### METRIC DEFINITIONS


Let:

- $B$ be the set of facilities in the best solution returned by the metaheuristic;
- $G_{top}$ be the weighted co-occurrence graph built from the top 5% LTM solutions;
- $A$ be its weighted adjacency matrix.

The benchmark uses the following definitions.

**Communities**

- **Weighted modularity**: standard graph modularity computed with edge weights.
- **Coverage**: fraction of total edge weight that stays inside the detected communities.
- **Best-facility concentration**: if $C^*$ is the community with largest overlap with $B$, then $|B \cap C^*| / |B|$.
- **Best-facility purity**: for the same $C^*$, $|B \cap C^*| / |C^*|$.

**Max-p-Cut**

- **Fraction cut**: cut weight divided by total graph weight.
- **Best-facility separation**: fraction of unordered pairs of best facilities that are placed in different Max-p-Cut blocks.

**K-core**

- **Core mass**: size of the highest k-core divided by the number of facilities.
- **Best-core recall**: fraction of best facilities that belong to the highest k-core.
- **Best-core precision**: fraction of highest-k-core facilities that belong to the best solution.

**Densest subgraph**

- **Average internal weighted degree**: average weighted degree inside the extracted densest subgraph.
- **Best overlap**: Jaccard overlap between the densest-subgraph node set and the best-facility set.

In [ ]:
def build_top_ltm(
    long_term_memory: list[dict[str, object]],
    top_fraction: float,
) -> tuple[list[dict[str, object]], np.ndarray, np.ndarray]:
    if not long_term_memory:
        raise ValueError("long_term_memory is empty.")

    top_solution_count = max(1, int(np.ceil(len(long_term_memory) * top_fraction)))

    analysis_ltm = sorted(long_term_memory, key=lambda sol: float(sol["cost"]))[:top_solution_count]

    matrix = np.vstack(
        [
            np.asarray(sol["facilities"], dtype=np.int8)
            for sol in analysis_ltm
        ]
    )

    costs = np.asarray(
        [
            float(sol["cost"])
            for sol in analysis_ltm
        ],
        dtype=float,
    )

    return analysis_ltm, matrix, costs


def build_cooccurrence_matrix(matrix: np.ndarray) -> np.ndarray:
    adjacency = np.asarray(matrix.T @ matrix, dtype=np.int64)
    np.fill_diagonal(adjacency, 0)

    return adjacency


def build_weighted_graph(adjacency: np.ndarray) -> nx.Graph:
    n = adjacency.shape[0]
    graph = nx.Graph()
    graph.add_nodes_from(range(n))

    rows, cols = np.where(np.triu(adjacency, k=1) > 0)

    for i, j in zip(rows.tolist(), cols.tolist()):
        graph.add_edge(int(i), int(j), weight=float(adjacency[i, j]))

    return graph


def build_unweighted_graph(adjacency: np.ndarray) -> nx.Graph:
    n = adjacency.shape[0]
    graph = nx.Graph()
    graph.add_nodes_from(range(n))

    rows, cols = np.where(np.triu(adjacency, k=1) > 0)
    graph.add_edges_from((int(i), int(j)) for i, j in zip(rows.tolist(), cols.tolist()))

    return graph


def total_edge_weight(adjacency: np.ndarray) -> float:
    return float(np.triu(adjacency, k=1).sum())


def community_internal_weight(adjacency: np.ndarray, nodes: set[int] | frozenset[int]) -> float:
    node_array = np.array(sorted(nodes), dtype=int)

    if node_array.size <= 1:
        return 0.0

    subgraph = adjacency[np.ix_(node_array, node_array)]

    return float(np.triu(subgraph, k=1).sum())


def weighted_coverage(
    adjacency: np.ndarray,
    communities_found: list[set[int] | frozenset[int]],
) -> float:
    total_weight = total_edge_weight(adjacency)

    if total_weight <= 0.0:
        return 0.0

    internal_weight = sum(
        community_internal_weight(adjacency, community_nodes)
        for community_nodes in communities_found
    )

    return internal_weight / total_weight


def select_best_overlap_community(
    communities_found: list[set[int] | frozenset[int]],
    adjacency: np.ndarray,
    best_set: set[int],
) -> set[int]:
    if not communities_found:
        return set()

    best_community = set()
    best_key = None

    for community_nodes in communities_found:
        community_set = set(community_nodes)
        overlap = len(community_set.intersection(best_set))
        size = len(community_set)
        internal_weight = community_internal_weight(adjacency, community_set)
        min_node = min(community_set) if community_set else 10**9

        key = (
            overlap,
            -size,
            internal_weight,
            -min_node,
        )

        if best_key is None or key > best_key:
            best_key = key
            best_community = community_set

    return best_community


def community_metrics(
    graph: nx.Graph,
    adjacency: np.ndarray,
    best_set: set[int],
) -> dict[str, float | int]:
    if graph.number_of_edges() == 0:
        communities_found = [frozenset([node]) for node in graph.nodes]
        weighted_modularity = 0.0
    else:
        communities_found = list(
            community.greedy_modularity_communities(graph, weight="weight")
        )
        weighted_modularity = float(
            nx.community.modularity(graph, communities_found, weight="weight")
        )

    target_community = select_best_overlap_community(
        communities_found,
        adjacency,
        best_set,
    )

    overlap = len(target_community.intersection(best_set))

    return {
        "communities_count": len(communities_found),
        "communities_weighted_modularity": weighted_modularity,
        "communities_coverage": weighted_coverage(adjacency, communities_found),
        "communities_best_facility_concentration": overlap / max(1, len(best_set)),
        "communities_best_facility_purity": overlap / max(1, len(target_community)),
    }


def _random_labels(n: int, p: int, rng: np.random.Generator) -> np.ndarray:
    labels = rng.integers(0, p, size=n)

    if n >= p:
        permutation = rng.permutation(n)
        labels[permutation[:p]] = np.arange(p)

    return labels


def _group_sums(A: np.ndarray, labels: np.ndarray, p: int) -> np.ndarray:
    sums = np.zeros((A.shape[0], p), dtype=float)

    for group_id in range(p):
        members = labels == group_id

        if np.any(members):
            sums[:, group_id] = A[:, members].sum(axis=1)

    return sums


def max_p_cut_local_search(
    A: np.ndarray,
    p: int,
    n_restarts: int = 30,
    max_iter: int = 2000,
    seed: int = 42,
) -> tuple[np.ndarray, float, float]:
    A = np.asarray(A, dtype=float)

    n = A.shape[0]
    p = max(1, min(int(p), n))

    total_weight = float(np.triu(A, 1).sum())

    if total_weight <= 1e-12:
        labels = np.arange(n, dtype=int) % p
        return labels, 0.0, 0.0

    rng = np.random.default_rng(seed)

    best_cut = -np.inf
    best_internal = np.inf
    best_labels = None

    for _ in range(max(1, int(n_restarts))):
        labels = _random_labels(n, p, rng)
        sums = _group_sums(A, labels, p)

        internal = float(np.triu(A * (labels[:, None] == labels[None, :]), 1).sum())
        cut = total_weight - internal

        for _ in range(max(1, int(max_iter))):
            improved = False

            for node in rng.permutation(n):
                old_group = labels[node]

                gains = sums[node, old_group] - sums[node, :]
                gains[old_group] = -np.inf

                new_group = int(np.argmax(gains))
                gain = float(gains[new_group])

                if gain > 1e-12:
                    labels[node] = new_group
                    cut += gain

                    sums[:, old_group] -= A[:, node]
                    sums[:, new_group] += A[:, node]

                    improved = True

            if not improved:
                break

        internal = total_weight - cut

        if cut > best_cut + 1e-12:
            best_cut = float(cut)
            best_internal = float(internal)
            best_labels = labels.copy()

    return best_labels, best_cut, best_internal


def best_facility_separation(labels: np.ndarray, best_set: set[int]) -> float:
    best_nodes = sorted(best_set)

    if len(best_nodes) <= 1:
        return 1.0

    separated_pairs = sum(
        labels[i] != labels[j]
        for i, j in combinations(best_nodes, 2)
    )

    return separated_pairs / math.comb(len(best_nodes), 2)


def k_core_metrics(
    graph: nx.Graph,
    n: int,
    best_set: set[int],
) -> dict[str, float | int]:
    core_numbers = nx.core_number(graph)
    max_core_level = max(core_numbers.values()) if core_numbers else 0

    highest_core_nodes = {
        node
        for node, core_level in core_numbers.items()
        if core_level == max_core_level
    }

    overlap = highest_core_nodes.intersection(best_set)

    return {
        "k_core_max_level": int(max_core_level),
        "k_core_core_size": len(highest_core_nodes),
        "k_core_core_mass": len(highest_core_nodes) / max(1, n),
        "k_core_best_core_recall": len(overlap) / max(1, len(best_set)),
        "k_core_best_core_precision": len(overlap) / max(1, len(highest_core_nodes)),
    }


def densest_subgraph_greedy(
    adjacency: np.ndarray,
    min_size: int = 3,
) -> tuple[set[int], float]:
    n = adjacency.shape[0]

    remaining = set(range(n))
    best_subgraph = set(range(n)) if n and n < min_size else set()
    best_density = -np.inf

    while len(remaining) >= max(1, min_size):
        nodes = sorted(remaining)
        subgraph = adjacency[np.ix_(nodes, nodes)]

        current_weight = float(np.triu(subgraph, k=1).sum())
        current_density = current_weight / max(1, len(remaining))

        if current_density > best_density:
            best_density = current_density
            best_subgraph = remaining.copy()

        min_pos = int(np.argmin(subgraph.sum(axis=1)))
        remaining.remove(nodes[min_pos])

    if not best_subgraph and n:
        best_subgraph = set(range(min(n, max(1, min_size))))
        best_density = 0.0

    return best_subgraph, max(0.0, float(best_density))


def average_internal_weighted_degree(
    adjacency: np.ndarray,
    nodes: set[int],
) -> float:
    if not nodes:
        return 0.0

    node_list = sorted(nodes)
    subgraph = adjacency[np.ix_(node_list, node_list)]
    weighted_degrees = subgraph.sum(axis=1)

    return float(np.mean(weighted_degrees))


def jaccard_overlap(set_a: set[int], set_b: set[int]) -> float:
    union = set_a.union(set_b)

    if not union:
        return 1.0

    return len(set_a.intersection(set_b)) / len(union)


def densest_subgraph_metrics(
    adjacency: np.ndarray,
    best_set: set[int],
) -> dict[str, float | int]:
    densest_nodes, _ = densest_subgraph_greedy(adjacency, min_size=3)

    return {
        "densest_subgraph_size": len(densest_nodes),
        "densest_subgraph_average_internal_weighted_degree": average_internal_weighted_degree(
            adjacency,
            densest_nodes,
        ),
        "densest_subgraph_best_overlap": jaccard_overlap(densest_nodes, best_set),
    }

In [ ]:
def run_single_instance_analysis(
    instance_name: str,
    *,
    restarts: int,
    max_iter: int,
    factor: int,
    top_fraction: float,
    details_format: str = "binary",
    max_p_cut_restarts: int = 30,
    max_p_cut_max_iter: int = 2000,
    global_seed: int = 42,
) -> dict[str, object]:
    instance_name = canonical_instance_name(instance_name)
    instance_path = INSTANCES_DIR / instance_name
    instance_id = Path(instance_name).stem

    row = {
        "instance": instance_name,
        "instance_id": instance_id,
        "n": np.nan,
        "p": np.nan,
        "best_known_cost": BEST_KNOWN_COSTS.get(instance_id, np.nan),
        "best_cost": np.nan,
        "gap_percent": np.nan,
        "memory_size": np.nan,
        "top_solution_count": np.nan,
        "top_cost_cutoff": np.nan,
        "cooccurrence_edges": np.nan,
        "cooccurrence_total_weight": np.nan,
        "metaheuristic_runtime_seconds": np.nan,
        "structure_extraction_runtime_seconds": np.nan,
        "total_pipeline_runtime_seconds": np.nan,
        "communities_count": np.nan,
        "communities_weighted_modularity": np.nan,
        "communities_coverage": np.nan,
        "communities_best_facility_concentration": np.nan,
        "communities_best_facility_purity": np.nan,
        "max_p_cut_fraction_cut": np.nan,
        "max_p_cut_best_facility_separation": np.nan,
        "k_core_max_level": np.nan,
        "k_core_core_size": np.nan,
        "k_core_core_mass": np.nan,
        "k_core_best_core_recall": np.nan,
        "k_core_best_core_precision": np.nan,
        "densest_subgraph_size": np.nan,
        "densest_subgraph_average_internal_weighted_degree": np.nan,
        "densest_subgraph_best_overlap": np.nan,
        "status": "error",
        "error_message": None,
    }

    if not instance_path.exists():
        row["error_message"] = f"Instance not found: {instance_path}"
        return row

    metadata = read_instance_metadata(instance_path)
    row["n"] = metadata["n"]
    row["p"] = metadata["p"]

    overall_started_at = perf_counter()

    try:
        meta_started_at = perf_counter()

        summary, details = pymedian.solve_pmedian(
            instance_path,
            restarts=restarts,
            max_iter=max_iter,
            factor=factor,
            details_format=details_format,
        )

        metaheuristic_runtime_seconds = perf_counter() - meta_started_at

        long_term_memory = details["long_term_memory"]
        if not long_term_memory:
            raise ValueError("long_term_memory is empty.")

        analysis_ltm, matrix, costs = build_top_ltm(long_term_memory, top_fraction)
        adjacency = build_cooccurrence_matrix(matrix)

        weighted_graph = build_weighted_graph(adjacency)
        unweighted_graph = build_unweighted_graph(adjacency)

        best_facilities = tuple(
            sorted(
                int(value) - 1
                for value in summary["tspmed_facilities"]
            )
        )
        best_set = set(best_facilities)

        structure_started_at = perf_counter()

        community_stats = community_metrics(weighted_graph, adjacency, best_set)

        labels_maxcut, cut_weight, internal_weight = max_p_cut_local_search(
            adjacency,
            int(summary["p"]),
            n_restarts=max_p_cut_restarts,
            max_iter=max_p_cut_max_iter,
            seed=global_seed,
        )

        max_p_cut_stats = {
            "max_p_cut_fraction_cut": cut_weight / max(1e-12, cut_weight + internal_weight),
            "max_p_cut_best_facility_separation": best_facility_separation(
                labels_maxcut,
                best_set,
            ),
        }

        k_core_stats = k_core_metrics(
            unweighted_graph,
            int(summary["n"]),
            best_set,
        )

        densest_stats = densest_subgraph_metrics(adjacency, best_set)

        structure_extraction_runtime_seconds = perf_counter() - structure_started_at
        total_pipeline_runtime_seconds = perf_counter() - overall_started_at

        row.update(
            {
                "n": int(summary["n"]),
                "p": int(summary["p"]),
                "best_cost": float(summary["tspmed_cost"]),
                "gap_percent": compute_gap_percent(
                    summary["tspmed_cost"],
                    row["best_known_cost"],
                ),
                "memory_size": len(long_term_memory),
                "top_solution_count": len(analysis_ltm),
                "top_cost_cutoff": float(costs.max()),
                "cooccurrence_edges": int(weighted_graph.number_of_edges()),
                "cooccurrence_total_weight": total_edge_weight(adjacency),
                "metaheuristic_runtime_seconds": metaheuristic_runtime_seconds,
                "structure_extraction_runtime_seconds": structure_extraction_runtime_seconds,
                "total_pipeline_runtime_seconds": total_pipeline_runtime_seconds,
                **community_stats,
                **max_p_cut_stats,
                **k_core_stats,
                **densest_stats,
                "status": "ok",
                "error_message": None,
            }
        )
    except Exception as exc:
        row["total_pipeline_runtime_seconds"] = perf_counter() - overall_started_at
        row["error_message"] = f"{type(exc).__name__}: {exc}"

    return row


def run_benchmark(
    instance_names: list[str],
    *,
    restarts: int,
    max_iter: int,
    factor: int,
    top_fraction: float,
    details_format: str = "binary",
    max_p_cut_restarts: int = 30,
    max_p_cut_max_iter: int = 2000,
    global_seed: int = 42,
) -> pd.DataFrame:
    if not instance_names:
        raise ValueError("The benchmark requires at least one instance.")

    rows = []

    for instance_name in tqdm(
        instance_names,
        total=len(instance_names),
        desc="Structural benchmark",
        dynamic_ncols=True,
    ):
        rows.append(
            run_single_instance_analysis(
                instance_name,
                restarts=restarts,
                max_iter=max_iter,
                factor=factor,
                top_fraction=top_fraction,
                details_format=details_format,
                max_p_cut_restarts=max_p_cut_restarts,
                max_p_cut_max_iter=max_p_cut_max_iter,
                global_seed=global_seed,
            )
        )

    results_df = pd.DataFrame(rows)

    results_df["interpretability_overhead_ratio"] = np.where(
        results_df["metaheuristic_runtime_seconds"] > 0,
        results_df["structure_extraction_runtime_seconds"]
        / results_df["metaheuristic_runtime_seconds"],
        np.nan,
    )

    results_df["instance_order"] = results_df["instance"].map(
        lambda value: instance_sort_key(value)[0]
    )

    return (
        results_df.sort_values(["instance_order", "instance"])
        .drop(columns="instance_order")
        .reset_index(drop=True)
    )

### APPLY

In [ ]:
results_df = run_benchmark(
    INSTANCE_NAMES,
    restarts=RESTARTS,
    max_iter=MAX_ITER,
    factor=FACTOR,
    top_fraction=TOP_FRACTION,
    details_format=DETAILS_FORMAT,
    max_p_cut_restarts=MAX_P_CUT_RESTARTS,
    max_p_cut_max_iter=MAX_P_CUT_MAX_ITER,
    global_seed=GLOBAL_SEED,
)

final_column_order = [
    "instance",
    "instance_id",
    "n",
    "p",
    "best_known_cost",
    "best_cost",
    "gap_percent",
    "memory_size",
    "top_solution_count",
    "top_cost_cutoff",
    "cooccurrence_edges",
    "cooccurrence_total_weight",
    "metaheuristic_runtime_seconds",
    "structure_extraction_runtime_seconds",
    "interpretability_overhead_ratio",
    "total_pipeline_runtime_seconds",
    "communities_count",
    "communities_weighted_modularity",
    "communities_coverage",
    "communities_best_facility_concentration",
    "communities_best_facility_purity",
    "max_p_cut_fraction_cut",
    "max_p_cut_best_facility_separation",
    "k_core_max_level",
    "k_core_core_size",
    "k_core_core_mass",
    "k_core_best_core_recall",
    "k_core_best_core_precision",
    "densest_subgraph_size",
    "densest_subgraph_average_internal_weighted_degree",
    "densest_subgraph_best_overlap",
    "status",
    "error_message",
]

final_results_df = results_df[final_column_order].copy()

display(final_results_df)

In [ ]:
success_df = final_results_df.loc[final_results_df["status"] == "ok"].copy()
failure_df = final_results_df.loc[
    final_results_df["status"] != "ok",
    ["instance", "instance_id", "status", "error_message"],
].reset_index(drop=True)

print(f"Successful instances : {len(success_df)}")
print(f"Failed instances     : {len(failure_df)}")

if SAVE_RESULTS_CSV:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    final_results_df.to_csv(RESULTS_CSV, index=False)
    print(f"Results saved to     : {RESULTS_CSV}")

if SAVE_FAILURES_CSV and not failure_df.empty:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    failure_df.to_csv(FAILURES_CSV, index=False)
    print(f"Failures saved to    : {FAILURES_CSV}")

if failure_df.empty:
    print("No execution failures were recorded.")
else:
    display(failure_df)